# New Arrivals Playlist

this is a notebook on the update process for the new arrivals playlist - it helps understanding the tracks table and the update steps.

In [ ]:
import os

os.chdir("..")
print(os.getcwd())

In [ ]:
import pandas as pd

## 0. Read user configuration

In [ ]:
from pydantic import BaseModel

In [ ]:
class Authentication(BaseModel):
    client_id: str
    client_secret: str
    redirect_uri: str
    """Spotify API authentication credentials"""


class CatalogItem(BaseModel):
    """this will be used for playlists and artists"""

    id: str
    url: str
    name: str
    allowed_tracks: int
    """how many tracks allowed for the new arrival playlist from this Item"""


class User(BaseModel):
    """Defines the user configuration *(which playlists/artists to consider, username, etc.)*"""

    id: str
    auth: Authentication
    spotify_user_name: str
    playlists: list[CatalogItem]
    artists: list[CatalogItem]

    @property
    def playlists_df(self) -> pd.DataFrame:
        """Convert playlists to DataFrame"""
        return pd.DataFrame([playlist.model_dump() for playlist in self.playlists])

    @property
    def artists_df(self) -> pd.DataFrame:
        """Convert artists to DataFrame"""
        return pd.DataFrame([artist.model_dump() for artist in self.artists])

    @classmethod
    def from_json(cls, json_path: str) -> "User":
        """Load user configuration from JSON file"""
        with open(json_path, "r") as f:
            return cls.model_validate_json(f.read())

    def to_json(self, json_path: str) -> None:
        """Save user configuration to JSON file"""
        os.makedirs(os.path.dirname(json_path), exist_ok=True)
        with open(json_path, "w") as f:
            f.write(self.model_dump_json(indent=2))

In [ ]:
DATABASE_FOLDER = "../../data/"

In [ ]:
with open(DATABASE_FOLDER + "config_test.json", "r") as f:
    user = User.model_validate_json(f.read())

## 1. read current tracks data

In [ ]:
df_tracks = pd.read_parquet(DATABASE_FOLDER + "latest_tracks.parquet")
df_tracks.head()

## 2. get latest favorites and add them

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth

In [ ]:
os.environ["SPOTIPY_CLIENT_ID"] = user.auth.client_id
os.environ["SPOTIPY_CLIENT_SECRET"] = user.auth.client_secret
os.environ["SPOTIPY_REDIRECT_URI"] = user.auth.redirect_uri

In [ ]:
# find the most recently added favorite track
idx = df_tracks.loc[df_tracks["added_from"] == "favorites", "date_added"].idxmax()
latest_fav_id = df_tracks.loc[idx, "id"]
print("latest favorite track id:", latest_fav_id)

In [ ]:
scope = "user-library-read"
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(scope=scope))

In [ ]:
# empirically we see, that the results are ordered by most recently added first
results = []
offset = 0

while True:
    page = sp.current_user_saved_tracks(limit=50, offset=offset)
    items = page["items"]
    if not items:
        print("no more items")
        break

    for item in items:
        tid = item["track"]["id"]
        if tid == latest_fav_id:
            break  # reached the point
        results.append(item)

    else:
        # runs only if the inner loop did NOT break
        offset += 50
        if offset >= 150:  # safety break
            raise RuntimeError(
                "Latest favorite not found within 150 tracks. "
                "State is inconsistent (track likely removed)."
            )

        continue

    break  # found latest_fav_id inside the page

In [ ]:
from datetime import date, datetime
from pydantic import BaseModel

In [ ]:
class Track(BaseModel):
    id: str
    name: str
    artist_ids: list[str]
    artist_names: list[str]
    date_added: date
    added_from: str
    """playlist id, artist id or favorites when added from favorites"""

In [ ]:
existing_ids = set(df_tracks["id"])

new_fav_tracks = [
    Track(
        id=track["track"]["id"],
        name=track["track"]["name"],
        artist_ids=[artist["id"] for artist in track["track"]["artists"]],
        artist_names=[artist["name"] for artist in track["track"]["artists"]],
        date_added=datetime.strptime(
            track["added_at"], "%Y-%m-%dT%H:%M:%SZ"
        ).date(),
        added_from="favorites",
    )
    for track in results if track["track"]["id"] not in existing_ids
]

## 3. get current state of playlists and artists

In [ ]:
today = date.today()
new_pla_tracks = []

for playlist in user.playlists:
    tracks_in_playlist = []
    tot_tracks = sp.playlist(playlist.id, fields="tracks.total")["tracks"]["total"]
    batch_size = 100

    for offset in range(0, tot_tracks, batch_size):
        tracks = sp.playlist_items(
            playlist.id,
            limit=batch_size,
            offset=offset,
            fields="items(track(id, popularity)",
        )["items"]
        tracks_in_playlist.extend([t["track"] for t in tracks])

    df_tracks_playlist = pd.DataFrame(tracks_in_playlist)
    df_tracks_playlist = df_tracks_playlist[
        ~df_tracks_playlist["id"].isin(df_tracks["id"])
    ]
    df_tracks_playlist.sort_values(by="popularity", ascending=False, inplace=True)
    df_tracks_playlist.drop(columns=["popularity"], inplace=True)
    df_tracks_playlist["added_from"] = f"pla:{playlist.id}"
    df_tracks_playlist["date_added"] = today
    if df_tracks_playlist.empty:
        print("playlist done (no new tracks):", playlist.name)
        continue

    new_pla_tracks.append(df_tracks_playlist.iloc[: playlist.allowed_tracks])
    print("playlist done:", playlist.name)

In [ ]:
new_art_tracks = []
for artist in user.artists:
    # just keep the relevant fields
    tracks_in_artist = [
        {
            "id": track["id"],
            "name": track["name"],
            "popularity": track["popularity"],
            "artist_ids": [artist["id"] for artist in track["artists"]],
            "artist_names": [artist["name"] for artist in track["artists"]],
        }
        for track in sp.artist_top_tracks(artist.id)["tracks"]
    ]

    df_tracks_artist = pd.DataFrame(tracks_in_artist)
    df_tracks_artist = df_tracks_artist[~df_tracks_artist["id"].isin(existing_ids)]
    df_tracks_artist.sort_values(by="popularity", ascending=False, inplace=True)
    df_tracks_artist.drop(columns=["popularity"], inplace=True)
    df_tracks_artist["added_from"] = f"art:{artist.id}"
    df_tracks_artist["date_added"] = today
    if df_tracks_artist.empty:
        print("artist done (no new tracks):", artist.name)
        continue
    
    new_art_tracks.append(df_tracks_artist.iloc[: artist.allowed_tracks])

    print("☑️ artist done:", artist.name)

## 4. Enrich df_tracks with new tracks from favorites, playlists and artists

In [ ]:
df_tracks_updated: pd.DataFrame = pd.concat(
    [df_tracks]
    + new_fav_tracks
    + new_pla_tracks
    + new_art_tracks,
    ignore_index=True,
)

# drop potential duplicates (in case that artist and playlist added the same track)
len_before = len(df_tracks_updated)
df_tracks_updated.drop_duplicates(subset=["id"], inplace=True)
len_after = len(df_tracks_updated)
if len_before != len_after:
    print(
        f"⚠️ Dropped {len_before - len_after} duplicate tracks when merging new tracks."
    )

In [ ]:
df_tracks_updated.tail()

## 5. Populate missing values

In [ ]:
def populate_null(row):
    if row.isnull().any():
        curr_track = sp.track(row["id"])
        row["name"] = curr_track["name"]
        row["artist_ids"] = [artist["id"] for artist in curr_track["artists"]]
        row["artist_names"] = [artist["name"] for artist in curr_track["artists"]]
        return row
    else:
        return row

In [ ]:
# NOTE: apply with axis=1 can be slow for large dataframes - but here it's acceptable
df_tracks_updated: pd.DataFrame = df_tracks_updated.apply(populate_null, axis=1)

## 6. save updated tracks table

In [ ]:
current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
df_tracks_updated.to_parquet(f"data/{current_time}_tracks.parquet", index=False)
df_tracks_updated.to_parquet("data/latest_tracks.parquet", index=False)

## 7. Create playlist for today's new arrivals

In [ ]:
# do the filtering
df_for_playlist = df_tracks_updated[df_tracks_updated["date_added"] == today]
df_for_playlist = df_for_playlist[df_for_playlist["added_from"] != "favorites"]

cnt_tracks = df_for_playlist.shape[0]
if cnt_tracks > 0:
    playlist = sp.user_playlist_create(
        user.spotify_user_name,
        f"New arrivals {today}",
        public=False,
        collaborative=False,
        description="New arrivals playlist",
    )
    ids_to_add = df_for_playlist["id"].tolist()
    batch_size = 100
    for i in range(0, len(ids_to_add), batch_size):
        sp.playlist_add_items(
            playlist["id"],
            ids_to_add[i : i + batch_size],
            position=i,
        )
    print(f"☑️ Playlist created with {cnt_tracks} tracks!")
else:
    print("ℹ️ No new tracks added today from playlists or artists.")